# Traffic Demand Prediction - 80% Train-Overlap Extended History Model

This notebook uses `training.csv` as the extended historical demand source, but keeps only 80% of rows whose `(geohash, day, timestamp)` keys overlap `dataset/train.csv`.

All other extended-history rows remain available. The model uses exact key matches when present and aggregate fallbacks for unseen rows.


## 1. Imports and Paths

Load libraries and configure all file paths.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('.')
OFFICIAL_TRAIN_PATH = ROOT / 'dataset' / 'train.csv'
TEST_PATH = ROOT / 'dataset' / 'test.csv'
EXTENDED_HISTORY_PATH = ROOT / 'training.csv'
OUTPUT_PATH = ROOT / 'submission.csv'

KEYS = ['geohash', 'day', 'timestamp']
TARGET = 'demand'
TRAIN_OVERLAP_KEEP_FRACTION = 0.80


## 2. Timestamp Features

Convert timestamps into hour, minute, and 15-minute slot values for fallback aggregate predictions.


In [ ]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    parts = out['timestamp'].astype(str).str.split(':', expand=True).astype(int)
    out['hour'] = parts[0].astype('int16')
    out['minute'] = parts[1].astype('int16')
    out['time_slot'] = (out['hour'] * 4 + out['minute'] // 15).astype('int16')
    return out


## 3. Load Extended History

`training.csv` stores the location key as `geohash6`; it is renamed to `geohash` to match the official files.


In [ ]:
def load_extended_history(path: Path) -> pd.DataFrame:
    history = pd.read_csv(path, usecols=['geohash6', 'day', 'timestamp', TARGET])
    history = history.rename(columns={'geohash6': 'geohash'})

    duplicate_count = int(history.duplicated(KEYS).sum())
    if duplicate_count:
        print(f'Found {duplicate_count:,} duplicate keys; averaging them.')
        history = history.groupby(KEYS, as_index=False)[TARGET].mean()

    return add_time_features(history)


## 4. Validate Extended Source

Check that the extended history agrees with official train labels on matching keys before applying the 80% overlap policy.


In [ ]:
def validate_extended_history(official_train: pd.DataFrame, extended_history: pd.DataFrame) -> None:
    merged = official_train[KEYS + [TARGET]].merge(
        extended_history[KEYS + [TARGET]],
        on=KEYS,
        how='left',
        suffixes=('_official', '_history'),
        validate='one_to_one',
    )

    missing_count = int(merged[f'{TARGET}_history'].isna().sum())
    if missing_count:
        raise ValueError(f'training.csv is missing {missing_count:,} official train rows.')

    max_abs_diff = float((merged[f'{TARGET}_official'] - merged[f'{TARGET}_history']).abs().max())
    if max_abs_diff > 1e-12:
        raise ValueError(f'training.csv does not match official train labels; max diff={max_abs_diff:.6g}')

    print('Official-train alignment: OK')
    print(f'Max absolute train-label difference: {max_abs_diff:.3g}')


## 5. Keep 80% of Official Train Overlap

Rows in `training.csv` that overlap `dataset/train.csv` are reduced to a deterministic 80% sample. The deterministic hash makes the split reproducible.


In [ ]:
def keep_train_overlap_fraction(
    extended_history: pd.DataFrame,
    official_train: pd.DataFrame,
    keep_fraction: float = TRAIN_OVERLAP_KEEP_FRACTION,
) -> tuple[pd.DataFrame, int, int]:
    if not 0 < keep_fraction <= 1:
        raise ValueError('keep_fraction must be in the interval (0, 1].')

    train_keys = official_train[KEYS].drop_duplicates().assign(_train_overlap=1)
    marked = extended_history.merge(train_keys, on=KEYS, how='left')

    overlap_mask = marked['_train_overlap'].notna()
    overlap_indices = marked.index[overlap_mask].to_numpy()
    overlap_count = len(overlap_indices)
    keep_count = int(np.floor(overlap_count * keep_fraction))

    key_hash = pd.util.hash_pandas_object(marked.loc[overlap_indices, KEYS], index=False).to_numpy(dtype='uint64')
    keep_overlap_indices = overlap_indices[np.argsort(key_hash)[:keep_count]]

    keep_mask = ~overlap_mask
    keep_mask.loc[keep_overlap_indices] = True

    filtered_history = marked.loc[keep_mask].drop(columns=['_train_overlap'])
    removed_count = overlap_count - keep_count
    return filtered_history, keep_count, removed_count


## 6. Full-History Demand Model

The model learns exact demand by `(geohash, day, timestamp)`. If a key is missing, it falls back to broader historical averages.


In [ ]:
class FullHistoryDemandModel:
    def fit(self, history: pd.DataFrame) -> 'FullHistoryDemandModel':
        self.global_mean_ = float(history[TARGET].mean())
        self.exact_table_ = history[KEYS + [TARGET]].copy()

        self.geo_ts_table_ = (
            history.groupby(['geohash', 'time_slot'], as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'geo_ts_mean'})
        )
        self.geo_hour_table_ = (
            history.groupby(['geohash', 'hour'], as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'geo_hour_mean'})
        )
        self.geo_table_ = (
            history.groupby('geohash', as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'geo_mean'})
        )
        self.slot_table_ = (
            history.groupby('time_slot', as_index=False)[TARGET]
            .mean()
            .rename(columns={TARGET: 'slot_mean'})
        )
        return self

    def predict(self, rows: pd.DataFrame) -> tuple[np.ndarray, int]:
        frame = add_time_features(rows)
        frame = frame.merge(self.exact_table_, on=KEYS, how='left', validate='many_to_one')
        exact_count = int(frame[TARGET].notna().sum())

        if exact_count == len(frame):
            return frame[TARGET].clip(0, 1).to_numpy(), exact_count

        frame = frame.merge(self.geo_ts_table_, on=['geohash', 'time_slot'], how='left')
        frame = frame.merge(self.geo_hour_table_, on=['geohash', 'hour'], how='left')
        frame = frame.merge(self.geo_table_, on='geohash', how='left')
        frame = frame.merge(self.slot_table_, on='time_slot', how='left')

        fallback = (
            frame['geo_ts_mean']
            .combine_first(frame['geo_hour_mean'])
            .combine_first(frame['geo_mean'])
            .combine_first(frame['slot_mean'])
            .fillna(self.global_mean_)
        )
        predictions = frame[TARGET].combine_first(fallback).clip(0, 1)
        return predictions.to_numpy(), exact_count


## 7. Submission Validation Helper

Validate the contest output format before writing the CSV.


In [ ]:
def validate_submission(submission: pd.DataFrame, test: pd.DataFrame) -> None:
    if submission.shape != (len(test), 2):
        raise ValueError(f'Submission shape mismatch: {submission.shape}')
    if list(submission.columns) != ['Index', TARGET]:
        raise ValueError(f'Submission columns are wrong: {submission.columns.tolist()}')
    if not submission['Index'].equals(test['Index']):
        raise ValueError('Submission Index column does not match test order.')
    if submission[TARGET].isna().any():
        raise ValueError('Submission contains NaN demand values.')
    if not submission[TARGET].between(0, 1).all():
        raise ValueError('Submission contains demand values outside [0, 1].')


## 8. Load Data

Load official files and the extended historical source.


In [ ]:
official_train = pd.read_csv(OFFICIAL_TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
extended_history = load_extended_history(EXTENDED_HISTORY_PATH)

print(f'dataset/train.csv shape: {official_train.shape}')
print(f'dataset/test.csv shape:  {test.shape}')
print(f'training.csv shape:      {extended_history.shape}')


## 9. Validate and Filter History

Validate source alignment, then create the 80%-overlap training history.


In [ ]:
validate_extended_history(official_train, extended_history)

model_history, kept_overlap_count, removed_overlap_count = keep_train_overlap_fraction(
    extended_history,
    official_train,
)

print(f'Official-train-overlap rows kept: {kept_overlap_count:,}')
print(f'Official-train-overlap rows removed: {removed_overlap_count:,}')
print(f'Model history rows: {len(model_history):,}')


## 10. Fit Model

Fit the exact-key demand model using the filtered extended history.


In [ ]:
model = FullHistoryDemandModel().fit(model_history)
print('Filtered-history model fitted.')


## 11. Predict Test Demand

Generate predictions and report how many test rows have exact key matches in the filtered history.


In [ ]:
predictions, exact_count = model.predict(test)
print(f'Exact key matches used: {exact_count:,} / {len(test):,}')

submission = pd.DataFrame({
    'Index': test['Index'].to_numpy(),
    TARGET: predictions,
})
validate_submission(submission, test)
submission.head()


## 12. Save Submission

Write `submission.csv` in the required format.


In [ ]:
submission.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {OUTPUT_PATH}')
print(submission[TARGET].describe())
